# Splitting, Nested CV, and Data Leakage — Lab
### BBBP (blood–brain barrier permeability) dataset

**What you will learn**
- Split chemical datasets using random, scaffold, and time-based strategies with RDKit
- Implement nested cross-validation to tune hyperparameters without touching the test set
- Plot learning curves and validation curves to diagnose underfitting and overfitting
- Detect and fix a data leakage bug in a preprocessing pipeline

**How to use this notebook:** run each cell in order. Cells marked **`### EXERCISE`**
have a docstring / hints and a `# YOUR CODE HERE` marker — fill those in. Everything
else is provided scaffolding so you can spend your time on the interesting parts.
Discussion questions are marked **`Discuss:`** — some have no single right answer, so
talk them through with a neighbour before you write anything down.

> Runtime once fully filled in: end‑to‑end this notebook takes **~4–5 minutes** on
> Colab's free CPU runtime. The nested CV and learning/validation curve cells are the
> slow ones (each ~15–45s) — that's normal, not a bug in your code.


## 1. Set up the environment

We need `rdkit` (cheminformatics: SMILES parsing, scaffolds, fingerprints), `scikit-learn`
(splitting, models, CV, metrics), and the usual `pandas`/`numpy`/`matplotlib`. This cell
is provided — just run it and confirm it prints `Setup OK.`


In [ ]:
# Colab: uncomment the line below on first run
# !pip install -q rdkit scikit-learn pandas matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, KFold, GridSearchCV,
    learning_curve, validation_curve,
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import roc_auc_score

RDLogger.DisableLog("rdApp.*")   # RDKit is chatty about odd valences/stereo in BBBP; silence it
RNG_SEED = 42
np.random.seed(RNG_SEED)

print("Setup OK.")


## 2. Explore the dataset

We pull the standard 2050-molecule BBBP table (Martins et al., 2012 — MoleculeNet's
canonical mirror). `p_np` is the label: **1 = penetrates the blood–brain barrier,
0 = does not.**

The load-and-clean cell below is provided — **read it carefully anyway**, because the
instructions ask you to *"confirm the dataset loads and SMILES parse cleanly,"* and
this exact mirror has a few rows worth noticing before they contaminate your splits.


In [ ]:
url = "https://raw.githubusercontent.com/GLambard/Molecules_Dataset_Collection/master/latest/BBBP.csv"
df_raw = pd.read_csv(url)
print(f"Raw rows: {len(df_raw)}")
df_raw.head()


In [ ]:
# --- Clean: drop missing SMILES, canonicalize, drop unparseable rows (provided) ---
df = df_raw.dropna(subset=["smiles"]).copy()
print(f"Dropped {len(df_raw) - len(df)} rows with missing SMILES -> {len(df)} remain")

def canonicalize(smi):
    mol = Chem.MolFromSmiles(smi)
    return Chem.MolToSmiles(mol) if mol is not None else None

df["canon_smiles"] = df["smiles"].apply(canonicalize)
n_fail = df["canon_smiles"].isna().sum()
print(f"SMILES that failed to parse: {n_fail}")
df = df.dropna(subset=["canon_smiles"]).reset_index(drop=True)


### `### EXERCISE` 2.1 — find duplicated structures

The instructions ask you to *"identify any duplicated SMILES strings."* Careful: you
should dedupe on the **canonical** SMILES, not the raw string — two different SMILES
strings can represent the identical molecule. Fill in the three lines below.

**Hints**
- `df["canon_smiles"].duplicated()` flags dupes on the canonical form.
- To find duplicate *groups*, `df[df.duplicated("canon_smiles", keep=False)]` keeps
  every row that shares a canonical SMILES with at least one other row.
- Within each duplicate group, check `.groupby("canon_smiles")["p_np"].nunique()` —
  a value of 2 means that "duplicate" has **conflicting labels**, which is a data
  quality problem you should resolve (e.g. keep the first occurrence) before splitting.


In [ ]:
### EXERCISE 2.1: duplicate detection
# TODO: count duplicated raw SMILES strings vs. duplicated CANONICAL SMILES, and count
# how many duplicate groups have conflicting p_np labels. Then resolve by keeping the
# first occurrence of each canonical structure.

# n_dupe_raw = ...
# n_dupe_canon = ...
# conflicts = ...   # Series: canon_smiles -> nunique(p_np), for duplicated rows only
# n_conflict = ...

# YOUR CODE HERE

print(f"Duplicated raw SMILES strings:      {n_dupe_raw}")
print(f"Duplicated CANONICAL SMILES:        {n_dupe_canon}")
print(f"Duplicate groups with CONFLICTING p_np labels: {n_conflict} / {len(conflicts)} duplicate groups")

# Resolve: keep first occurrence of each canonical structure
df = df.drop_duplicates("canon_smiles").reset_index(drop=True)
print(f"Final clean dataset: {len(df)} unique molecules")


In [ ]:
# --- Class balance (provided) ---
class_counts = df["p_np"].value_counts().sort_index()
pos_rate = df["p_np"].mean()
print(class_counts)
print(f"BBB-positive fraction: {pos_rate:.3f}")

fig, ax = plt.subplots(figsize=(4, 3.5))
ax.bar(["BBB-negative (0)", "BBB-positive (1)"], class_counts.values,
       color=["#4C72B0", "#DD8452"])
ax.set_ylabel("Number of molecules")
ax.set_title(f"BBBP class balance (n={len(df)})")
for i, v in enumerate(class_counts.values):
    ax.text(i, v + 15, str(v), ha="center")
plt.tight_layout()
plt.show()


> **Discuss — Warm-up:** What fraction of molecules are BBB-positive? Would you expect
> this class imbalance to affect AUC-ROC? What about accuracy? *(Think about what a
> classifier that always predicts "positive" would score on each metric.)*
>
> *Your notes:*


## 3. Implement splitting strategies

We featurize once (Morgan/ECFP4 fingerprints — 1024-bit, radius 2) and reuse `X`, `y`
across every split and model below. The featurizer is provided.


In [ ]:
def featurize_morgan(smiles_list, n_bits=1024, radius=2):
    X = np.zeros((len(smiles_list), n_bits), dtype=int)
    for i, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smi)
        bv = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
        DataStructs.ConvertToNumpyArray(bv, X[i])
    return X

X = featurize_morgan(df["canon_smiles"])
y = df["p_np"].values
print(X.shape, y.shape)


### `### EXERCISE` 3a — random split

Use `sklearn.model_selection.train_test_split` to make an 80/20 split. Stratify on
`y` so both sides keep roughly the same class balance. Store the **indices**, not the
arrays themselves (`np.arange(len(df))` is your input array to split) — later cells
index into `X`/`y` with these.


In [ ]:
### EXERCISE 3a: random split
# TODO: produce train_idx_rand, test_idx_rand (arrays of row indices, 80/20, stratified)

# YOUR CODE HERE

print(f"Random split: {len(train_idx_rand)} train / {len(test_idx_rand)} test")
print(f"  train pos rate: {y[train_idx_rand].mean():.3f}   test pos rate: {y[test_idx_rand].mean():.3f}")


### `### EXERCISE` 3b — scaffold split (Bemis–Murcko)

The idea: molecules sharing a scaffold are chemically similar, so if the *same*
scaffold appears in both train and test, the model can succeed by memorizing
scaffold-level patterns rather than generalizing. A proper scaffold split puts every
molecule with a given scaffold **entirely** in one bucket — never split across train
and test.

**Two functions to implement:**

1. `get_murcko_scaffold(smi)` — return the Bemis–Murcko scaffold SMILES for a
   molecule. Use `rdkit.Chem.Scaffolds.MurckoScaffold.MurckoScaffoldSmiles`. Molecules
   with no ring system return an empty string from that function — in that case, fall
   back to using the molecule's own canonical SMILES as its "scaffold" (it's its own
   bucket of one).
2. `scaffold_split(scaffolds, frac_train=0.8, frac_valid=0.1, seed=...)` — bucket
   molecule indices by scaffold, then **greedily pack whole scaffold groups** into
   train/valid/test in that priority order until each hits its size target. Pack
   **largest groups first** (this mirrors `dc.splits.ScaffoldSplitter`) but **shuffle
   before sorting** to break ties reproducibly — without the shuffle, the single
   largest 2–3 scaffolds can dominate one split and skew class balance badly.

**Hints**
- `collections.defaultdict(list)` is a natural structure for "scaffold -> list of row indices".
- `list(dict.values())` gives you the groups; `random.Random(seed).shuffle(groups)`
  then `groups.sort(key=len, reverse=True)` gives shuffled-tie-break, largest-first order.
- Track running totals (`len(train_idx)`, etc.) against `frac_train * n` and
  `(frac_train + frac_valid) * n` cutoffs as you iterate through groups.


In [ ]:
from collections import defaultdict
import random as _random

def get_murcko_scaffold(smi):
    """Return the Bemis-Murcko scaffold SMILES for a molecule.
    Falls back to the molecule's own SMILES if it has no ring system."""
    # YOUR CODE HERE
    raise NotImplementedError

df["scaffold"] = df["canon_smiles"].apply(get_murcko_scaffold)
print(f"Unique scaffolds: {df['scaffold'].nunique()} across {len(df)} molecules")


In [ ]:
### EXERCISE 3b: scaffold split
def scaffold_split(scaffolds, frac_train=0.8, frac_valid=0.1, seed=RNG_SEED):
    """Bucket indices by scaffold, then greedily pack whole scaffold groups into
    train/valid/test (largest groups first, shuffled for tie-breaking).
    Returns (train_idx, valid_idx, test_idx) as numpy arrays."""
    # YOUR CODE HERE
    raise NotImplementedError

train_idx_scaf, valid_idx_scaf, test_idx_scaf = scaffold_split(df["scaffold"].values)
print(f"Scaffold split: {len(train_idx_scaf)} train / {len(valid_idx_scaf)} valid / {len(test_idx_scaf)} test")
print(f"  train pos rate: {y[train_idx_scaf].mean():.3f}   test pos rate: {y[test_idx_scaf].mean():.3f}")

# Sanity check (provided) -- if this fails, your scaffold_split has a bug
train_scaf = set(df['scaffold'].values[train_idx_scaf])
test_scaf = set(df['scaffold'].values[test_idx_scaf])
assert len(train_scaf & test_scaf) == 0, "Leakage: a scaffold appears in both train and test!"
print("Scaffold-overlap check passed: 0 scaffolds shared between train and test.")

# --- DeepChem equivalent, if you'd rather use the library version to check your work: ---
# import deepchem as dc
# dataset = dc.data.DiskDataset.from_numpy(X=X, y=y, ids=df['canon_smiles'].values)
# splitter = dc.splits.ScaffoldSplitter()
# train_ds, valid_ds, test_ds = splitter.train_valid_test_split(dataset)


**What does a scaffold group actually look like?** (provided) Here's the largest
*non-trivial* scaffold in BBBP (plain benzene is technically the single most common
scaffold, but it's shared for a boring reason — lots of unrelated molecules happen to
contain an unsubstituted phenyl ring). This one is a real chemical series: a
corticosteroid core shared by dozens of related steroid drugs.


In [ ]:
from rdkit.Chem import Draw, AllChem

scaffold_counts = df["scaffold"].value_counts()
# Skip benzene itself (index 0) - grab the largest scaffold with a real ring system attached
target_scaffold = scaffold_counts.index[1]
print(f"Scaffold: {target_scaffold}   ({scaffold_counts.iloc[1]} molecules share it)")

members = df[df["scaffold"] == target_scaffold].sample(5, random_state=7)
mols = [Chem.MolFromSmiles(target_scaffold)] + [Chem.MolFromSmiles(s) for s in members["canon_smiles"]]
for m in mols:
    AllChem.Compute2DCoords(m)
legends = ["Shared scaffold\n(this is what stays together)"] + [
    f"{name}\nBBB={'+' if p else '-'}" for name, p in zip(members["name"], members["p_np"])
]
img = Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(240, 200), legends=legends)
img


Notice these are all clearly the same drug family — and in BBBP they even tend to
share the same label. **That's exactly the leakage risk a random split creates**: if
three of these five end up in train and two in test, the model doesn't need to learn
*why* steroids cross the blood-brain barrier — it just needs to recognize "I've seen
this exact scaffold before, and it was BBB-positive." Scaffold splitting forces every
member of a family into the same bucket, so that shortcut isn't available.


### `### EXERCISE` 3c — temporal split (by year)

**Important caveat:** the public BBBP table does **not** ship a per-molecule
publication year — Martins et al. (2012) compiled it from many older sources without
retaining dates. So there is no honest way to do a *real* temporal split on this
specific dataset.

To still practice the *mechanics* of a temporal split, the cell below **simulates** a
plausible `sim_year` column for you (provided — don't change this part). Your job is
just the split itself: **sort molecules by `sim_year` ascending, and put the earliest
80% in train, the most recent 20% in test.** In your own work, replace `sim_year` with
a real timestamp column and the same split code applies unchanged.

**Hint:** `np.argsort(..., kind="stable")` gives you an index array sorted by year;
slice it at `int(0.8 * len(df))`.


In [ ]:
# Simulated year column (provided; deterministic, seeded from scaffold so related
# molecules cluster in time). Do not edit this part.
rng = np.random.default_rng(RNG_SEED)
scaffold_years = {s: rng.integers(1995, 2013) for s in df["scaffold"].unique()}
df["sim_year"] = [scaffold_years[s] + rng.integers(-1, 2) for s in df["scaffold"]]

### EXERCISE 3c: sort by sim_year and split 80/20 (earliest -> train, latest -> test)
# TODO: produce train_idx_time, test_idx_time

# YOUR CODE HERE

print(f"Temporal split: {len(train_idx_time)} train (<= {df['sim_year'].iloc[train_idx_time].max()})"
      f" / {len(test_idx_time)} test ({df['sim_year'].iloc[test_idx_time].min()}+)")
print(f"  train pos rate: {y[train_idx_time].mean():.3f}   test pos rate: {y[test_idx_time].mean():.3f}")


### 3d. Compare class distributions across the three splits (provided)

In [ ]:
splits_summary = pd.DataFrame({
    "split": ["Random"] * 2 + ["Scaffold"] * 2 + ["Temporal (simulated)"] * 2,
    "subset": ["train", "test"] * 3,
    "pos_rate": [
        y[train_idx_rand].mean(), y[test_idx_rand].mean(),
        y[train_idx_scaf].mean(), y[test_idx_scaf].mean(),
        y[train_idx_time].mean(), y[test_idx_time].mean(),
    ],
})
print(splits_summary)

fig, ax = plt.subplots(figsize=(6, 4))
width = 0.35
splits = ["Random", "Scaffold", "Temporal (simulated)"]
train_rates = splits_summary[splits_summary.subset == "train"]["pos_rate"].values
test_rates = splits_summary[splits_summary.subset == "test"]["pos_rate"].values
x = np.arange(len(splits))
ax.bar(x - width/2, train_rates, width, label="train", color="#4C72B0")
ax.bar(x + width/2, test_rates, width, label="test", color="#DD8452")
ax.axhline(pos_rate, ls="--", color="gray", lw=1, label="overall")
ax.set_xticks(x); ax.set_xticklabels(splits)
ax.set_ylabel("BBB-positive rate")
ax.set_title("Class balance: train vs. test, by split strategy")
ax.legend()
plt.tight_layout()
plt.show()


**Check your work:** does the scaffold split's test pos-rate drift further from the
overall rate than the random split's does? If so, that's the split working as
intended, not a bug — keep it in mind for the next section.


### 3e. Visualize the splits in chemical space (PCA) — provided

Class-balance bar charts show *one* number per split. A PCA projection of the
fingerprints lets us see the *shape* of train vs. test in chemical space directly —
this is the plot that actually shows *why* scaffold splitting is the harder,
more honest evaluation.

**Honesty check on the method:** Morgan fingerprints are long, sparse, binary vectors,
so 2 principal components only capture a small slice of their total variance (printed
below — expect single digits of percent per axis). Treat this plot as a qualitative
sketch of chemical space, not a rigorous distance metric.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

pca = PCA(n_components=2, random_state=RNG_SEED)
X_pca = pca.fit_transform(X)
print(f"Explained variance: PC1={pca.explained_variance_ratio_[0]*100:.1f}%  "
      f"PC2={pca.explained_variance_ratio_[1]*100:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)
for ax, (title, tr_idx, te_idx) in zip(
    axes,
    [("Random split", train_idx_rand, test_idx_rand), ("Scaffold split", train_idx_scaf, test_idx_scaf)],
):
    ax.scatter(X_pca[tr_idx, 0], X_pca[tr_idx, 1], s=10, alpha=0.4, color="#4C72B0", label="train")
    ax.scatter(X_pca[te_idx, 0], X_pca[te_idx, 1], s=14, alpha=0.8, color="#DD8452", label="test")
    ax.set_title(title)
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
axes[0].legend(loc="upper right", fontsize=9)
plt.suptitle("Morgan fingerprint chemical space: train vs. test, by split strategy", y=1.02)
plt.tight_layout()
plt.show()

# Quantify what the eye is trying to judge: how far is each test molecule from its
# nearest TRAINING neighbour? A bigger number = more genuinely "novel" test set.
def mean_nn_dist(train_idx, test_idx, space):
    nn = NearestNeighbors(n_neighbors=1).fit(space[train_idx])
    d, _ = nn.kneighbors(space[test_idx])
    return d.mean()

print(f"Mean distance from each test molecule to its nearest training neighbour"
      f" (full {X.shape[1]}-d fingerprint space, not just the 2D PCA plot):")
print(f"  Random split:   {mean_nn_dist(train_idx_rand, test_idx_rand, X.astype(float)):.3f}")
print(f"  Scaffold split: {mean_nn_dist(train_idx_scaf, test_idx_scaf, X.astype(float)):.3f}")


**Check your work:** the scaffold-split panel should look visibly patchier — test
points (orange) clumping into their own pockets rather than sitting uniformly inside
the train cloud the way they do under random splitting. The nearest-neighbour numbers
should back that up: scaffold-split test molecules should sit noticeably farther from
their nearest training example than random-split test molecules do. That's the
concrete, geometric version of the AUC gap you're about to measure in Section 4.


## 4. Tune with nested cross-validation

**Why "nested"?** If you use the same data to (a) pick hyperparameters and (b) report
final performance, your reported score is optimistic — you've implicitly fit the
hyperparameters *to* the test set. Nested CV keeps them separate:

- **Outer loop:** one held-out test fold, touched **only once**, at the very end, for
  scoring.
- **Inner loop:** `GridSearchCV` over the *outer-training* portion only, to pick
  `n_estimators`, `max_depth`, `min_samples_leaf`.

### `### EXERCISE` 4 — fill in the grid and the nested CV function

1. Fill in `PARAM_GRID` with a small grid over the three hyperparameters named above
   (2–3 values each is plenty — remember every extra combination multiplies your
   runtime).
2. Complete `nested_cv_auc`: build a `StratifiedKFold` inner CV, wrap a
   `RandomForestClassifier` in `GridSearchCV` (`scoring="roc_auc"`), **fit it on
   `train_idx` only**, then score **once** on `test_idx`.

Write it as a function that takes `train_idx`/`test_idx` as arguments — that way you
can point it at *any* split (random or scaffold) with no code changes, which is
exactly what you need to answer the discussion question below.

⏱️ *Each call takes ~15–25s once correctly implemented.*


In [ ]:
### EXERCISE 4: hyperparameter grid
# TODO: pick 2-3 values for each hyperparameter
PARAM_GRID = {
    "n_estimators": [...],      # e.g. a couple of values, larger = slower but usually better
    "max_depth": [...],         # include None (unlimited) as one option
    "min_samples_leaf": [...],  # controls how "smooth" each tree's decision boundary is
}


In [ ]:
### EXERCISE 4: nested CV function
def nested_cv_auc(X, y, train_idx, test_idx, param_grid=PARAM_GRID, inner_folds=5, seed=RNG_SEED):
    """Inner-loop grid search on train_idx only; single honest score on test_idx.
    Returns (test_auc, best_params)."""
    # YOUR CODE HERE
    raise NotImplementedError

auc_random, params_random = nested_cv_auc(X, y, train_idx_rand, test_idx_rand)
print(f"[Random split]   outer test AUC-ROC: {auc_random:.4f}   best params: {params_random}")

auc_scaffold, params_scaffold = nested_cv_auc(X, y, train_idx_scaf, test_idx_scaf)
print(f"[Scaffold split] outer test AUC-ROC: {auc_scaffold:.4f}   best params: {params_scaffold}")


> **Discuss — Core:** Compare test AUC under random vs. scaffold splitting. Which is
> lower? By how much? What does this gap imply for model deployment? *(Think about
> what kind of molecule a deployed model actually gets asked to score — is it more
> like your random-split test set, or your scaffold-split test set?)*
>
> *Your notes:*


## 5. Diagnose your model: learning curve & validation curve

We use the **scaffold split's training set** (`train_idx_scaf`) for these diagnostics
— Section 4 told us scaffold generalization is the harder, more realistic problem, so
that's the regime worth understanding.

### `### EXERCISE` 5a — learning curve

Use `sklearn.model_selection.learning_curve` with a `RandomForestClassifier`
(`n_estimators=200` is a reasonable fixed choice here — we're studying data size, not
re-tuning), 5-fold `StratifiedKFold`, `scoring="roc_auc"`, and
`train_sizes=np.linspace(0.1, 1.0, 6)`. Plot training AUC and validation AUC vs.
training size on the same axes.

⏱️ *~30–45s.*


In [ ]:
### EXERCISE 5a: learning curve
X_work, y_work = X[train_idx_scaf], y[train_idx_scaf]

# TODO: call learning_curve(...) to get train_sizes, train_scores, val_scores
# YOUR CODE HERE

# TODO: plot train_scores.mean(axis=1) and val_scores.mean(axis=1) vs. train_sizes
# YOUR CODE HERE

print("Train AUC (last point):", round(train_scores.mean(axis=1)[-1], 4))
print("Val AUC   (last point):", round(val_scores.mean(axis=1)[-1], 4))


### `### EXERCISE` 5b — validation curve

Use `sklearn.model_selection.validation_curve` with the same model family, sweeping
`max_depth` over `[2, 4, 6, 8, 10, 15, 20, None]`. Plot training AUC and validation AUC
vs. `max_depth`.

⏱️ *~30–45s.*


In [ ]:
### EXERCISE 5b: validation curve
depths = [2, 4, 6, 8, 10, 15, 20, None]

# TODO: call validation_curve(...) to get train_scores2, val_scores2
# YOUR CODE HERE

# TODO: plot both curves vs. depth (cast depths to str for the x-axis, since None isn't numeric)
# YOUR CODE HERE


> **Discuss — Core:** Using both curves, identify the bias-variance regime your model
> is in. Where does the learning curve suggest more data would help most? Where does
> the validation curve suggest `max_depth` stops being useful?
>
> *Your notes:*


## 6. Find and fix the leakage bug

Below is a preprocessing helper in the style you'd often find copy-pasted into a
notebook. **Read it carefully before writing any code** — the hint from the
instructions is *"look at where `StandardScaler` is fit."*

```python
# --- preprocessing (as handed to you) ---
scaler = StandardScaler().fit(X)          # fit on ALL of X...
X_scaled = scaler.transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=0, stratify=y
)                                          # ...THEN split.
```

### `### EXERCISE` 6 — implement both the buggy and fixed versions, and compare

Complete `compare_buggy_vs_fixed` below:
- **buggy branch:** fit the scaler on `X_train` and `X_test` stacked together
  (`np.vstack`), then transform both.
- **fixed branch:** fit the scaler on `X_train` only, then transform both `X_train`
  and `X_test` with that same fitted scaler.
- Train `model_fn()` on the transformed training data and score AUC on the
  transformed test data, for both branches.


In [ ]:
### EXERCISE 6: buggy vs. fixed comparison
def compare_buggy_vs_fixed(X, y, model_fn, test_size=0.2, seed=0, scaler_cls=StandardScaler):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=seed, stratify=y
    )

    # TODO buggy branch: scaler fit on train+test combined
    # auc_buggy = ...

    # TODO fixed branch: scaler fit on train only
    # auc_fixed = ...

    # YOUR CODE HERE

    return auc_buggy, auc_fixed

# Try it with the RandomForest we've used all notebook:
auc_b, auc_f = compare_buggy_vs_fixed(X, y, lambda: RandomForestClassifier(
    n_estimators=200, random_state=RNG_SEED, n_jobs=-1))
print(f"RandomForest  ->  buggy AUC: {auc_b:.6f}   fixed AUC: {auc_f:.6f}   diff: {auc_b - auc_f:+.6f}")


> **Discuss:** What do you notice about the two numbers above? Before you conclude the
> bug "doesn't matter," think about *why* a RandomForest's predictions might be
> unaffected by exactly this kind of preprocessing change (hint: think about what a
> decision tree split actually looks at — the *order* of a feature's values, or its
> absolute scale?). Then try the same comparison with a different kind of model below
> and see if your explanation holds up.


In [ ]:
### EXERCISE 6b: does the bug matter for other models?
# TODO: call compare_buggy_vs_fixed with KNeighborsClassifier(n_neighbors=15)
# TODO: call compare_buggy_vs_fixed with LogisticRegression(max_iter=2000)

# YOUR CODE HERE


**Bonus (provided, for context)** — a more dangerous variant of "fit before split":
*supervised* feature selection using labels from the full dataset, not just an
unsupervised scaler. This is the classic "double-dipping" mistake from
microarray/genomics ML (Ambroise & McLachlan, 2002). We shrink to a small sample here
purely to make the effect visually dramatic — the same mechanism operates (more
quietly) at full dataset size too.


In [ ]:
import warnings

X_small, _, y_small, _ = train_test_split(X, y, train_size=150, random_state=0, stratify=y)
Xtr_s, Xte_s, ytr_s, yte_s = train_test_split(X_small, y_small, test_size=0.33, random_state=0, stratify=y_small)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    # BUGGY: SelectKBest sees train+test labels before picking features
    sel_buggy = SelectKBest(f_classif, k=20).fit(np.vstack([Xtr_s, Xte_s]), np.concatenate([ytr_s, yte_s]))
    auc_buggy_fs = roc_auc_score(
        yte_s, LogisticRegression(max_iter=2000).fit(sel_buggy.transform(Xtr_s), ytr_s)
                                                 .predict_proba(sel_buggy.transform(Xte_s))[:, 1])

    # FIXED: SelectKBest only ever sees the training labels
    sel_fixed = SelectKBest(f_classif, k=20).fit(Xtr_s, ytr_s)
    auc_fixed_fs = roc_auc_score(
        yte_s, LogisticRegression(max_iter=2000).fit(sel_fixed.transform(Xtr_s), ytr_s)
                                                 .predict_proba(sel_fixed.transform(Xte_s))[:, 1])

print(f"Supervised feature-selection leak (n=150) -> buggy AUC: {auc_buggy_fs:.4f}   "
      f"fixed AUC: {auc_fixed_fs:.4f}   diff: {auc_buggy_fs - auc_fixed_fs:+.4f}")


**Takeaway to internalize:** anything that gets `.fit()`-ed — scaler, imputer,
selector, PCA, target encoder — belongs *inside* the cross-validation loop, fit on the
training fold only, every time, no exceptions. `sklearn.pipeline.Pipeline` makes this
the default rather than something you have to remember:

```python
from sklearn.pipeline import Pipeline
pipe = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=2000))])
# pipe.fit(X_train, y_train) now correctly fits the scaler on X_train only
```

> **Discuss — Core:** Identify the leakage bug, in your own words, and report how test
> AUC changed for each model you tried. Why did the size of the effect differ between
> models?
>
> *Your notes:*


## 7. Challenge: stratified scaffold split

Plain scaffold splitting (Section 3b) packs the *largest* scaffold groups in first —
it doesn't care about class balance while doing so, so different folds can end up with
noticeably different positive rates.

### `### EXERCISE` 7 — extend the greedy packer to also balance class rate

Complete `balanced_scaffold_folds(scaffolds, y, k=5, pos_weight=0.5, seed=...)`:

- Bucket by scaffold and shuffle+sort groups largest-first, same as Section 3b.
- For each group, and for each candidate fold `f`, compute a **cost** that combines
  (a) how far that fold's size would be from the target size `len(scaffolds)/k`, and
  (b) how far that fold's positive rate would be from the overall positive rate
  `y.mean()`, weighted by `pos_weight`.
- Assign the group to whichever fold minimizes that cost.
- Return a list of `k` index lists.

**Hint:** `pos_weight=0` should reduce this to a plain size-balanced scaffold k-fold
(no class-balance term) — a good way to sanity check your implementation before adding
the class-balance term.

⏱️ *~30–45s (5-fold RF CV, evaluated twice, once your function is correct).*


In [ ]:
### EXERCISE 7: stratified scaffold folds
def balanced_scaffold_folds(scaffolds, y, k=5, pos_weight=0.5, seed=RNG_SEED):
    """Greedily assign whole scaffold groups to k folds, balancing both fold SIZE
    and fold POSITIVE RATE. pos_weight=0 reduces to a size-only scaffold k-fold;
    pos_weight>0 also stratifies by class. Returns a list of k index lists."""
    # YOUR CODE HERE
    raise NotImplementedError

def cv_auc_for_folds(X, y, folds, model_fn=lambda: RandomForestClassifier(
        n_estimators=200, random_state=RNG_SEED, n_jobs=-1)):
    """Provided: evaluate a model with each fold as the held-out test set in turn."""
    aucs = []
    all_idx = np.arange(len(y))
    for i in range(len(folds)):
        test_idx = np.array(folds[i])
        train_idx = np.setdiff1d(all_idx, test_idx)
        model = model_fn().fit(X[train_idx], y[train_idx])
        aucs.append(roc_auc_score(y[test_idx], model.predict_proba(X[test_idx])[:, 1]))
    return np.array(aucs)

scaffolds_arr = df["scaffold"].values
folds_unstrat = balanced_scaffold_folds(scaffolds_arr, y, k=5, pos_weight=0.0)
folds_strat = balanced_scaffold_folds(scaffolds_arr, y, k=5, pos_weight=0.5)

print("Unstratified scaffold folds — size / pos rate:")
for i, f in enumerate(folds_unstrat):
    print(f"  fold {i}: n={len(f):4d}   pos_rate={y[f].mean():.3f}")

print("\nStratified scaffold folds — size / pos rate:")
for i, f in enumerate(folds_strat):
    print(f"  fold {i}: n={len(f):4d}   pos_rate={y[f].mean():.3f}")


In [ ]:
# Provided: compare AUC distributions once your fold assignment is implemented
aucs_unstrat = cv_auc_for_folds(X, y, folds_unstrat)
aucs_strat = cv_auc_for_folds(X, y, folds_strat)

print(f"Unstratified scaffold CV: AUC = {aucs_unstrat.mean():.4f} +/- {aucs_unstrat.std():.4f}"
      f"   (per-fold: {np.round(aucs_unstrat, 3)})")
print(f"Stratified scaffold CV:   AUC = {aucs_strat.mean():.4f} +/- {aucs_strat.std():.4f}"
      f"   (per-fold: {np.round(aucs_strat, 3)})")

fig, ax = plt.subplots(figsize=(5, 4))
ax.boxplot([aucs_unstrat, aucs_strat])
ax.set_xticks([1, 2])
ax.set_xticklabels(["Unstratified\nscaffold", "Stratified\nscaffold"])
ax.scatter([1]*5, aucs_unstrat, alpha=0.6, color="#4C72B0")
ax.scatter([2]*5, aucs_strat, alpha=0.6, color="#DD8452")
ax.set_ylabel("Per-fold AUC-ROC")
ax.set_title("Fold-to-fold AUC spread: scaffold CV")
plt.tight_layout(); plt.show()


> **Discuss — Challenge:** How does the AUC distribution across folds compare to an
> unstratified scaffold split? Which would you trust more if you were using CV AUC to
> decide whether a new featurization is actually an improvement?
>
> *Your notes:*


## Wrap-up: questions to hand in / discuss

1. **Warm-up —** What fraction of molecules are BBB-positive? Would this class
   imbalance affect AUC-ROC? What about accuracy?
2. **Core —** Compare test AUC under random vs. scaffold splitting. Which is lower? By
   how much? What does this gap imply for model deployment?
3. **Core —** Identify the data leakage bug (hint: look at where `StandardScaler` is
   fit). Fix it and report how test AUC changes — for RandomForest, and for at least
   one other model type.
4. **Challenge —** How does the AUC distribution across folds compare between a
   stratified and an unstratified scaffold split?

### Suggested extensions, if you finish early
- Swap Morgan fingerprints for RDKit physicochemical descriptors (or concatenate both)
  and see whether the random-vs-scaffold AUC gap changes.
- Replace the simulated `sim_year` column with a real ChEMBL assay-date query for a
  target of your choice, and redo the temporal split for real.
- Try `dc.splits.ScaffoldSplitter` / `dc.splits.RandomSplitter` from DeepChem directly
  and confirm they roughly reproduce your Section 3 splits.
